# 🧠 Step 3: Train Multi-Task U-Net (Colab Pro GPU)
### Deep Neural Architecture Engineered to Push Accuracy to ~0.85

**Core Advantages over pixel-wise Gradient Boosting:**
1. **Full 2D Spatial Receptive Field:** Integrates contextual building morphology and shadow features.
2. **Multi-Task Synergistic Heads:** Mask loss + continuous coverage regression + boundary edge separation.
3. **Tversky Loss with Recall Bias:** (α=0.3, β=0.7) heavily penalizes missed buildings without drowning in false positives.
4. **Balanced Patch Sampling & D4 Augmentation:** Flips, 90° rotations, and radiometric jitter.

In [ ]:
# 1. Setup Environment, GPU & Libraries
import os, sys, time

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = "/content/drive/MyDrive/Encroachment"
    if not os.path.isdir(ROOT_DIR):
        ROOT_DIR = "/content/drive/My Drive/Encroachment"
    sys.path.append(ROOT_DIR)
    %pip install -q segmentation-models-pytorch torch torchvision rasterio geopandas
else:
    ROOT_DIR = os.getcwd()
    sys.path.append(ROOT_DIR)

import torch
import numpy as np
from encroach.config import Config
from encroach.data import make_dataloaders
from encroach.models import build_model
from encroach.losses import MultiTaskLoss

cfg = Config(ROOT=ROOT_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)")

In [ ]:
# 2. Build Dataloaders with Positive Patch Oversampling
loader_tr, loader_va = make_dataloaders(
    patch_dir=cfg.PATCH_DIR,
    batch_size=cfg.BATCH,
    pos_share=cfg.POS_PATCH_SHARE,
    num_workers=cfg.NUM_WORKERS
)
print(f"Batches per epoch: Train = {len(loader_tr)}, Val = {len(loader_va)}")

In [ ]:
# 3. Build Multi-Task Model, Optimizer & Loss Function
model = build_model(cfg).to(device)
criterion = MultiTaskLoss(cfg).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 4. Training Loop with Validation IoU / F1 Tracking and Early Stopping
best_val_iou = 0.0
patience_counter = 0
checkpoint_path = cfg.P("best_unet_model.pth")

print("Starting training for", cfg.EPOCHS, "epochs...")
for epoch in range(1, cfg.EPOCHS + 1):
    t_ep = time.time()
    model.train()
    tr_loss = 0.0
    
    for batch in loader_tr:
        imgs = batch["image"].to(device)
        batch_gpu = {k: v.to(device) for k, v in batch.items() if k != "image"}
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            preds = model(imgs)
            loss, _ = criterion(preds, batch_gpu)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
        
    scheduler.step()
    tr_loss /= max(len(loader_tr), 1)
    
    # Validation Phase
    model.eval()
    va_loss = 0.0
    val_tp, val_fp, val_fn = 0, 0, 0
    
    with torch.no_grad():
        for batch in loader_va:
            imgs = batch["image"].to(device)
            batch_gpu = {k: v.to(device) for k, v in batch.items() if k != "image"}
            with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                preds = model(imgs)
                loss, _ = criterion(preds, batch_gpu)
            va_loss += loss.item()
            
            pred_mask = (torch.sigmoid(preds["mask"]) >= 0.5)
            true_mask = (batch_gpu["mask"] == 1)
            valid_pix = (~batch_gpu["ignore_cls"])
            
            val_tp += (pred_mask & true_mask & valid_pix).sum().item()
            val_fp += (pred_mask & (~true_mask) & valid_pix).sum().item()
            val_fn += ((~pred_mask) & true_mask & valid_pix).sum().item()
            
    va_loss /= max(len(loader_va), 1)
    val_prec = val_tp / max(val_tp + val_fp, 1)
    val_rec = val_tp / max(val_tp + val_fn, 1)
    val_f1 = 2 * val_prec * val_rec / max(val_prec + val_rec, 1e-9)
    val_iou = val_tp / max(val_tp + val_fp + val_fn, 1)
    
    print(f"Epoch {epoch:02d}/{cfg.EPOCHS:02d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | "
          f"Val F1: {val_f1:.3f} (P: {val_prec:.3f}, R: {val_rec:.3f}) | Val IoU: {val_iou:.3f} | {time.time() - t_ep:.0f}s")
          
    # Save best checkpoint
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> New best validation IoU: {best_val_iou:.4f} saved to {checkpoint_path}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= cfg.PATIENCE:
            print(f"Early stopping triggered after {cfg.PATIENCE} epochs without improvement.")
            break

print("Training finished. Best model checkpoint saved to:", checkpoint_path)